Install the required packages

In [ ]:
!pip install -q "transformers==4.57.1" "datasets" "openpyxl" "scikit-learn" "accelerate"

In [ ]:
import torch
import transformers

print("PyTorch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Transformers: 4.57.1
CUDA available: True
GPU: Tesla T4


Upload the trained RoBERTa model only

In [ ]:
from google.colab import files

print("Upload your previously trained RoBERTa model ZIP file.")

uploaded_model = files.upload()

model_zip_name = next(iter(uploaded_model.keys()))

print("\nModel uploaded successfully:")
print(model_zip_name)

Upload your previously trained RoBERTa model ZIP file.


Saving roberta_updated_model.zip to roberta_updated_model.zip

Model uploaded successfully:
roberta_updated_model.zip


Extract and locate the trained RoBERTa

In [ ]:
import os
import zipfile
from pathlib import Path

model_extract_path = Path("/content/roberta_trained_model")
model_extract_path.mkdir(parents=True, exist_ok=True)

with zipfile.ZipFile(
    model_zip_name,
    "r"
) as zip_ref:
    zip_ref.extractall(model_extract_path)

# Find the folder containing config.json
model_path = None

for root, dirs, files_list in os.walk(model_extract_path):
    if "config.json" in files_list:
        model_path = Path(root)
        break

if model_path is None:
    raise FileNotFoundError(
        "Could not find config.json inside the uploaded ZIP."
    )

print("RoBERTa model extracted successfully.")
print("Model path:", model_path)

print("\nModel files:")
for file in sorted(model_path.iterdir()):
    print("-", file.name)

RoBERTa model extracted successfully.
Model path: /content/roberta_trained_model

Model files:
- config.json
- merges.txt
- model.safetensors
- special_tokens_map.json
- tokenizer.json
- tokenizer_config.json
- training_args.bin
- vocab.json


Upload the updated XLSX dataset

In [ ]:
from google.colab import files

print("Upload your updated XLSX dataset.")

uploaded_dataset = files.upload()

dataset_file_name = next(iter(uploaded_dataset.keys()))

print("\nDataset uploaded successfully:")
print(dataset_file_name)

Upload your updated XLSX dataset.


Saving Model final dataset.xlsx to Model final dataset (1).xlsx

Dataset uploaded successfully:
Model final dataset (1).xlsx


Inspect the updated dataset

In [ ]:
import pandas as pd

dataset_df = pd.read_excel(
    dataset_file_name
)

print("Dataset shape:", dataset_df.shape)

print("\nColumns:")
print(dataset_df.columns.tolist())

print("\nFirst 5 rows:")
display(dataset_df.head())

print("\nLabel distribution:")
print(dataset_df["Labels"].value_counts().sort_index())

print("\nMissing values:")
print(dataset_df.isnull().sum())

Dataset shape: (15046, 2)

Columns:
['Sentences', 'Labels']

First 5 rows:


,Sentences,Labels
0,"We were missing a dwarf in the team, and now w...",NaN
1,useless guy,0.0
2,You are so rude!,0.0
3,Where does the goat live?,0.0
4,Why are you barking like a dog?,0.0



Label distribution:
Labels
0.0    3010
1.0    3029
2.0    2999
3.0    3003
4.0    3000
Name: count, dtype: int64

Missing values:
Sentences    1
Labels       5
dtype: int64


In [ ]:
# Keep the original/adversarial boundary BEFORE cleaning
original_df = dataset_df.iloc[:15000].copy()
adversarial_df = dataset_df.iloc[15000:].copy()

print("Original rows before cleaning:", len(original_df))
print("Adversarial rows:", len(adversarial_df))

# Clean original dataset
original_df = original_df.dropna(
    subset=["Sentences", "Labels"]
).copy()

original_df["Sentences"] = (
    original_df["Sentences"]
    .astype(str)
    .str.strip()
)

original_df = original_df[
    original_df["Sentences"] != ""
].reset_index(drop=True)

original_df["Labels"] = (
    original_df["Labels"]
    .astype(int)
)

# Clean adversarial rows
adversarial_df = adversarial_df.dropna(
    subset=["Sentences", "Labels"]
).copy()

adversarial_df["Labels"] = (
    adversarial_df["Labels"]
    .astype(int)
)

adversarial_df = adversarial_df.reset_index(drop=True)

print("\nOriginal valid rows:", len(original_df))
print("Adversarial valid rows:", len(adversarial_df))

print("\nOriginal label distribution:")
print(original_df["Labels"].value_counts().sort_index())

print("\nAdversarial label distribution:")
print(adversarial_df["Labels"].value_counts().sort_index())

Original rows before cleaning: 15000
Adversarial rows: 46

Original valid rows: 14994
Adversarial valid rows: 46

Original label distribution:
Labels
0    2999
1    2999
2    2998
3    2999
4    2999
Name: count, dtype: int64

Adversarial label distribution:
Labels
0    11
1    30
3     4
4     1
Name: count, dtype: int64


Remove the single missing sentence

In [ ]:
# Remove the one row where the sentence is missing
dataset_df = dataset_df.dropna(
    subset=["Sentences"]
).reset_index(drop=True)

print("Dataset shape after removing missing sentence:", dataset_df.shape)

print("\nLabel distribution:")
print(
    dataset_df["Labels"]
    .value_counts()
    .sort_index()
)

Dataset shape after removing missing sentence: (15018, 2)

Label distribution:
Labels
0    3005
1    3010
2    2999
3    3003
4    3001
Name: count, dtype: int64


Shuffle and create a new stratified split

In [ ]:
from sklearn.model_selection import train_test_split

# Features and labels
X = dataset_df["Sentences"]
y = dataset_df["Labels"]

# First split:
# 80% training, 20% temporary
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.20,
    stratify=y,
    random_state=42,
    shuffle=True
)

# Second split:
# Temporary 20% -> 10% validation + 10% testing
X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.50,
    stratify=y_temp,
    random_state=42,
    shuffle=True
)

print("Dataset split completed.\n")

print("Training samples  :", len(X_train))
print("Validation samples:", len(X_val))
print("Testing samples   :", len(X_test))

print("\nTraining label distribution:")
print(y_train.value_counts().sort_index())

print("\nValidation label distribution:")
print(y_val.value_counts().sort_index())

print("\nTesting label distribution:")
print(y_test.value_counts().sort_index())

ValueError: Input y contains NaN.

Create Hugging Face datasets

In [ ]:
from datasets import Dataset

train_dataset = Dataset.from_dict({
    "text": X_train.tolist(),
    "labels": y_train.tolist()
})

val_dataset = Dataset.from_dict({
    "text": X_val.tolist(),
    "labels": y_val.tolist()
})

test_dataset = Dataset.from_dict({
    "text": X_test.tolist(),
    "labels": y_test.tolist()
})

print("Hugging Face datasets created successfully.\n")

print("Training dataset :", train_dataset)
print("Validation dataset:", val_dataset)
print("Testing dataset  :", test_dataset)

Hugging Face datasets created successfully.

Training dataset : Dataset({
    features: ['text', 'labels'],
    num_rows: 12014
})
Validation dataset: Dataset({
    features: ['text', 'labels'],
    num_rows: 1502
})
Testing dataset  : Dataset({
    features: ['text', 'labels'],
    num_rows: 1502
})


Load the tokenizer from your already-trained RoBERTa

In [ ]:
from transformers import AutoTokenizer

tokenizer = AutoTokenizer.from_pretrained(
    model_path
)

print("Tokenizer loaded successfully.")
print("Vocabulary size:", tokenizer.vocab_size)

Tokenizer loaded successfully.
Vocabulary size: 50265


Tokenize the train, validation, and test datasets

In [ ]:
def tokenize_function(batch):
    return tokenizer(
        batch["text"],
        truncation=True,
        padding="max_length",
        max_length=128
    )


print("Tokenizing training dataset...")
train_dataset = train_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenizing validation dataset...")
val_dataset = val_dataset.map(
    tokenize_function,
    batched=True
)

print("Tokenizing testing dataset...")
test_dataset = test_dataset.map(
    tokenize_function,
    batched=True
)


# Remove the original text column
train_dataset = train_dataset.remove_columns(["text"])
val_dataset = val_dataset.remove_columns(["text"])
test_dataset = test_dataset.remove_columns(["text"])


print("\nTokenization completed successfully!")

print("\nTraining columns:")
print(train_dataset.column_names)

print("\nValidation columns:")
print(val_dataset.column_names)

print("\nTesting columns:")
print(test_dataset.column_names)

Tokenizing training dataset...


Map:   0%|          | 0/12014 [00:00<?, ? examples/s]

Tokenizing validation dataset...


Map:   0%|          | 0/1502 [00:00<?, ? examples/s]

Tokenizing testing dataset...


Map:   0%|          | 0/1502 [00:00<?, ? examples/s]


Tokenization completed successfully!

Training columns:
['labels', 'input_ids', 'attention_mask']

Validation columns:
['labels', 'input_ids', 'attention_mask']

Testing columns:
['labels', 'input_ids', 'attention_mask']


Load your previously trained RoBERTa model

In [ ]:
import torch
from transformers import AutoModelForSequenceClassification

id2label = {
    0: "Insult",
    1: "Neutral",
    2: "Racism",
    3: "Sexual Abuse",
    4: "Threat"
}

label2id = {
    "Insult": 0,
    "Neutral": 1,
    "Racism": 2,
    "Sexual Abuse": 3,
    "Threat": 4
}

model = AutoModelForSequenceClassification.from_pretrained(
    model_path,
    num_labels=5,
    id2label=id2label,
    label2id=label2id
)

print("Previously trained RoBERTa loaded successfully.")
print("Number of labels:", model.config.num_labels)
print("Label mapping:", model.config.id2label)

print("\nGPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

Previously trained RoBERTa loaded successfully.
Number of labels: 5
Label mapping: {0: 'Insult', 1: 'Neutral', 2: 'Racism', 3: 'Sexual Abuse', 4: 'Threat'}

GPU available: True
GPU: Tesla T4


Define evaluation metrics

In [ ]:
import numpy as np
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)

    accuracy = accuracy_score(labels, predictions)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "macro_f1": f1
    }

print("Evaluation metrics configured successfully.")

Evaluation metrics configured successfully.


Configure continued fine-tuning

In [ ]:
from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="/content/roberta_updated_checkpoints",

    # Match original RoBERTa epoch count
    num_train_epochs=3,

    # Conservative LR for continued fine-tuning
    learning_rate=1e-5,

    # Batch sizes
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,

    # Regularization
    weight_decay=0.01,
    warmup_ratio=0.1,

    # Evaluation and checkpointing
    eval_strategy="epoch",
    save_strategy="epoch",

    # Keep best checkpoint
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,

    # T4 GPU
    fp16=True,

    # Logging
    logging_steps=100,
    save_total_limit=2,
    report_to="none",

    # Reproducibility
    seed=42
)

print("Training configuration created successfully.")
print("Epochs:", training_args.num_train_epochs)
print("Learning rate:", training_args.learning_rate)
print("Training batch size:", training_args.per_device_train_batch_size)
print("Evaluation batch size:", training_args.per_device_eval_batch_size)
print("FP16:", training_args.fp16)

Training configuration created successfully.
Epochs: 3
Learning rate: 1e-05
Training batch size: 16
Evaluation batch size: 32
FP16: True


Create the Trainer

In [ ]:
from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    processing_class=tokenizer,
    compute_metrics=compute_metrics
)

print("Trainer recreated successfully.")
print("Epochs:", trainer.args.num_train_epochs)
print("Learning rate:", trainer.args.learning_rate)
print("Device:", trainer.args.device)

Trainer recreated successfully.
Epochs: 3
Learning rate: 1e-05
Device: cuda:0


Start continued fine-tuning

In [ ]:
train_result = trainer.train()

print("\nContinued fine-tuning completed successfully.")

Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,Macro F1
1,0.049700,0.043256,0.993342,0.993333,0.993336,0.993330
2,0.017600,0.057495,0.992676,0.992695,0.992671,0.992673
3,0.004600,0.046953,0.993342,0.993351,0.993336,0.993335



Continued fine-tuning completed successfully.


Evaluate on the test set

In [ ]:
test_results = trainer.evaluate(test_dataset)

print("\nTest Results:")
for key, value in test_results.items():
    if isinstance(value, float):
        print(f"{key}: {value:.6f}")
    else:
        print(f"{key}: {value}")


Test Results:
eval_loss: 0.061434
eval_accuracy: 0.991345
eval_precision: 0.991342
eval_recall: 0.991342
eval_macro_f1: 0.991340
eval_runtime: 3.000300
eval_samples_per_second: 500.609000
eval_steps_per_second: 15.665000
epoch: 3.000000


Classification report + confusion matrix

In [ ]:
import numpy as np
import pandas as pd
from sklearn.metrics import classification_report, confusion_matrix

# Get predictions on test dataset
pred_output = trainer.predict(test_dataset)

y_true = pred_output.label_ids
y_pred = np.argmax(pred_output.predictions, axis=-1)

class_names = [
    "Insult",
    "Neutral",
    "Racism",
    "Sexual Abuse",
    "Threat"
]

print("CLASSIFICATION REPORT\n")

print(
    classification_report(
        y_true,
        y_pred,
        target_names=class_names,
        digits=4,
        zero_division=0
    )
)

# Confusion matrix
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=[0, 1, 2, 3, 4]
)

cm_df = pd.DataFrame(
    cm,
    index=class_names,
    columns=class_names
)

print("\nCONFUSION MATRIX")
display(cm_df)

CLASSIFICATION REPORT

              precision    recall  f1-score   support

      Insult     0.9901    0.9934    0.9917       301
     Neutral     0.9934    0.9934    0.9934       301
      Racism     0.9833    0.9800    0.9816       300
Sexual Abuse     0.9934    0.9967    0.9950       300
      Threat     0.9967    0.9933    0.9950       300

    accuracy                         0.9913      1502
   macro avg     0.9913    0.9913    0.9913      1502
weighted avg     0.9913    0.9913    0.9913      1502


CONFUSION MATRIX


,Insult,Neutral,Racism,Sexual Abuse,Threat
Insult,299,0,1,1,0
Neutral,0,299,2,0,0
Racism,2,2,294,1,1
Sexual Abuse,1,0,0,299,0
Threat,0,0,2,0,298


Save the updated RoBERTa model

In [ ]:
updated_model_path = "/content/roberta_updated_model"

trainer.save_model(updated_model_path)
tokenizer.save_pretrained(updated_model_path)

print("Updated RoBERTa model saved successfully.")
print("Saved to:", updated_model_path)

Updated RoBERTa model saved successfully.
Saved to: /content/roberta_updated_model


Zip and download the updated model

In [ ]:
import shutil
from google.colab import files

zip_path = "/content/roberta_updated_model"

shutil.make_archive(
    zip_path,
    "zip",
    updated_model_path
)

print("ZIP created successfully:")
print(zip_path + ".zip")

files.download(zip_path + ".zip")

ZIP created successfully:
/content/roberta_updated_model.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>